# The Kernel Trick

**Companion lesson:** https://ml-viz.vercel.app/courses/svm/02-kernel-trick

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Kernel Functions

In [ ]:
x1 = np.array([0, 0])
x2 = np.array([1, 1])

def linear_kernel(x1, x2): return x1 @ x2
def poly_kernel(x1, x2, c=1, d=2): return (x1 @ x2 + c)**d
def rbf_kernel(x1, x2, gamma=1): return np.exp(-gamma * np.linalg.norm(x1 - x2)**2)

print(f'Linear: {linear_kernel(x1, x2)}')
print(f'Poly (d=2): {poly_kernel(x1, x2)}')
print(f'RBF (γ=1): {rbf_kernel(x1, x2):.4f}')
print(f'RBF (γ=0.1): {rbf_kernel(x1, x2, gamma=0.1):.4f}')

## Linear vs RBF: Non-linearly separable data

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import SVC

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    svm = SVC(kernel=kernel, C=1.0, gamma='scale')
    svm.fit(X, y)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=15, alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=15, alpha=0.7)
    ax.set_title(f'{kernel.capitalize()} Kernel', color='white')
plt.suptitle('SVM with Different Kernels', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## The kernel trick: implicit feature maps

A kernel $K(x, x')$ computes a dot product in a high-dimensional space **without ever building it**. The RBF kernel corresponds to an infinite-dimensional space.

In [ ]:
# Explicit degree-2 map vs the polynomial kernel give the same dot product
def phi(x):  # map (a, b) -> (a^2, b^2, sqrt(2) a b, sqrt(2) a, sqrt(2) b, 1)
    a, b = x
    return np.array([a**2, b**2, np.sqrt(2)*a*b, np.sqrt(2)*a, np.sqrt(2)*b, 1])

x1, x2 = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print('explicit  phi(x1).phi(x2):', phi(x1) @ phi(x2))
print('kernel    (x1.x2 + 1)^2 :', (x1 @ x2 + 1)**2)

In [ ]:
# Mercer's condition: a valid kernel has a symmetric POSITIVE SEMI-DEFINITE
# Gram matrix on every finite point set (all eigenvalues >= 0).
import numpy as np
rng = np.random.default_rng(0)
P = rng.normal(size=(6, 2))

def gram(K):
    return np.array([[K(a, b) for b in P] for a in P])

lin = lambda a, b: a @ b
poly = lambda a, b: (a @ b + 1) ** 2
rbf = lambda a, b: np.exp(-0.5 * np.sum((a - b) ** 2))
sig = lambda a, b: np.tanh(1.0 * (a @ b) + 1.0)   # NOT generally a valid kernel

for name, K in [('linear', lin), ('poly d=2', poly), ('rbf', rbf), ('sigmoid', sig)]:
    G = gram(K)
    eig = np.linalg.eigvalsh((G + G.T) / 2)        # symmetric part
    psd = np.all(eig >= -1e-8)
    print(f'{name:9s}: min eigenvalue = {eig.min():+.3f}  -> PSD (valid kernel)? {psd}')


## gamma controls RBF reach

Small `gamma` = smooth, far-reaching influence; large `gamma` = tight, wiggly boundaries that can overfit.

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.2, random_state=1)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(axes, [0.1, 1, 30]):
    svm = SVC(kernel='rbf', gamma=g, C=1).fit(X, y)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(*X.T, c=y, cmap='RdYlBu', edgecolor='k', s=15)
    ax.set_title(f'gamma = {g}')
plt.tight_layout(); plt.show()

## Key takeaways

- Kernels let a linear SVM learn **non-linear** boundaries via implicit feature maps.
- **RBF** is the go-to general-purpose kernel; **polynomial** and **linear** are alternatives.
- `C` controls margin softness; `gamma` controls RBF reach — tune both together (grid search).
- Large `gamma` or `C` overfits; always scale features before kernel SVMs.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The RBF kernel

The radial basis function kernel scores similarity by distance:

$$k(\mathbf{x}, \mathbf{z}) = \exp\!\big(-\gamma \, \lVert \mathbf{x} - \mathbf{z} \rVert^2\big)$$

Implement it. The checks verify the properties that make it a kernel — $k(\mathbf{x}, \mathbf{x}) = 1$, symmetry — and the $\gamma$ behavior from the section above: bigger $\gamma$ makes similarity die faster with distance.

In [ ]:
def rbf(x, z, gamma=1.0):
    """RBF (Gaussian) kernel between two vectors."""
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)

    # TODO(you): squared Euclidean distance between x and z
    sq_dist = ...

    # TODO(you): exp(-gamma * squared distance)
    return ...

In [ ]:
# Checks — run me
assert abs(rbf([1, 2], [1, 2]) - 1.0) < 1e-12, "a point is perfectly similar to itself"
assert abs(rbf([0, 0], [1, 0], gamma=1.0) - np.exp(-1)) < 1e-12, "unit distance, gamma=1 -> e^-1"
assert abs(rbf([1, 3], [2, 5]) - rbf([2, 5], [1, 3])) < 1e-15, "kernels are symmetric"
assert rbf([0, 0], [1, 0], gamma=10.0) < rbf([0, 0], [1, 0], gamma=0.1), \
    "bigger gamma -> similarity dies faster with distance"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def rbf(x, z, gamma=1.0):
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    sq_dist = np.sum((x - z) ** 2)
    return np.exp(-gamma * sq_dist)
```

</details>

### Exercise 2 — The kernel trick, verified

The quadratic kernel $k(\mathbf{x}, \mathbf{z}) = (\mathbf{x} \cdot \mathbf{z})^2$ secretly computes a dot product in 3D feature space:

$$\varphi(\mathbf{x}) = \big(x_1^2, \; \sqrt{2}\, x_1 x_2, \; x_2^2\big)
\qquad\Rightarrow\qquad
k(\mathbf{x}, \mathbf{z}) = \varphi(\mathbf{x}) \cdot \varphi(\mathbf{z})$$

Implement both sides and let the checks confirm they agree on random inputs — the kernel gets the 3D answer **without ever building the 3D vectors**.

In [ ]:
def poly2_kernel(x, z):
    """Quadratic kernel (x . z)^2 — never leaves 2D."""
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)

    # TODO(you): square of the ordinary dot product
    return ...


def phi(x):
    """The explicit 3D feature map (x1^2, sqrt(2) x1 x2, x2^2)."""
    x1, x2 = float(x[0]), float(x[1])

    # TODO(you): build the 3-vector
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
for _ in range(5):
    x, z = rng.standard_normal(2), rng.standard_normal(2)
    assert abs(poly2_kernel(x, z) - np.dot(phi(x), phi(z))) < 1e-9, \
        "(x.z)^2 must equal the dot product in the explicit 3D feature space"

assert abs(poly2_kernel([1, 2], [3, 1]) - 25.0) < 1e-12, "(1*3 + 2*1)^2 = 25"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def poly2_kernel(x, z):
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    return float(np.dot(x, z) ** 2)


def phi(x):
    x1, x2 = float(x[0]), float(x[1])
    return np.array([x1 ** 2, np.sqrt(2) * x1 * x2, x2 ** 2])
```

</details>